In [ ]:
import torch
import torch.nn as nn
from transformers import AutoModel
from torch.utils.data import Dataset

def collate_fn(batch):
    input_ids = torch.stack([item['input_ids'] for item in batch])
    attention_mask = torch.stack([item['attention_mask'] for item in batch])
    targets = torch.stack([item['target'] for item in batch])
    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "target": targets
    }

class TitleDataset(Dataset):
    def __init__(self, df, tokenizer, target_col="target", max_length=32):
        self.targets = df[target_col].to_list()
        self.encodings = tokenizer(
            df["video_title"].to_list(),
            padding="max_length",
            truncation=True,
            max_length=max_length,
            return_tensors="pt"
        )

    def __len__(self):
        return len(self.targets)

    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.encodings.items()}
        item["target"] = torch.tensor(self.targets[idx]).float()
        return item

class TitleRegressor(nn.Module):
    def __init__(self, model_name, dropout_rate=0.2, freeze_n_layers=None):
        super().__init__()
        self.dropout_rate = dropout_rate
        self.encoder = AutoModel.from_pretrained(model_name)

        print(self.encoder.config)

        if freeze_n_layers is not None:
            self._freeze_layers(freeze_n_layers)
        
        hidden_dim = self.encoder.config.hidden_size
        self.regressor = nn.Sequential(
            nn.Linear(hidden_dim, 128),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(128, 1)  # Output layer for regression
        )

    def _freeze_layers(self, n):
        # Freeze embeddings if present
        if hasattr(self.encoder, 'embeddings'):
            for param in self.encoder.embeddings.parameters():
                param.requires_grad = False

        # Try locating layer stack via known patterns
        candidate_paths = [
            ("encoder.layer", getattr(self.encoder, "encoder", None)),
            ("transformer.layer", getattr(self.encoder, "transformer", None)),
            ("distilbert.transformer.layer", getattr(getattr(self.encoder, "distilbert", None), "transformer", None)),
            ("layers", self.encoder),  # ModernBERT: layers at top-level
        ]

        for name, module in candidate_paths:
            if module and hasattr(module, "layer"):
                layers = getattr(module, "layer")
                if isinstance(layers, nn.ModuleList):
                    break
            elif isinstance(module, nn.Module) and hasattr(module, "layers"):
                layers = getattr(module, "layers")
                if isinstance(layers, nn.ModuleList):
                    break
        else:
            # Fallback: search recursively for first ModuleList with enough layers
            layers = next(
                (m for m in self.encoder.modules() if isinstance(m, nn.ModuleList) and len(m) >= n),
                None
            )
            if layers is None:
                raise ValueError("Cannot locate transformer layers in the model")

        # Cap n at available number of layers
        n = min(n, len(layers))
        print(f"Freezing first {n} of {len(layers)} layers.")

        for layer in layers[:n]:
            for param in layer.parameters():
                param.requires_grad = False


    def forward(self, input_ids, attention_mask):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        cls_embedding = outputs.last_hidden_state[:, 0, :]  # CLS token
        return self.regressor(cls_embedding)

    def summary(self):
        print(f"TitleRegressor with {self.encoder.config.hidden_size} hidden size and {len(self.regressor)} layers.")
        total, trainable = 0, 0
        for name, param in self.named_parameters():  # fixed from `model.named_parameters()`
            total += param.numel()
            if param.requires_grad:
                trainable += param.numel()
                print(f"Trainable: {name} - {param.numel()} params")
        print()
        print(f"Total parameters: {total:,}")
        print(f"Trainable parameters: {trainable:,}")
        print(f"Percentage of trainable parameters: {trainable / total:.2%}")
        
        effective_parameters = int(total * (1 - self.dropout_rate) // 1)
        print(f"Effective parameters after dropout: {effective_parameters:,}")

        effective_trainable = int(trainable * (1 - self.dropout_rate) // 1)
        print(f"Effective trainable parameters after dropout: {effective_trainable:,}")

from sklearn.model_selection import train_test_split

from enum import Enum
# Model configuration
class ModelType(Enum):
    BERT_TINY = "prajjwal1/bert-tiny"
    DISTILBERT_BASE = "distilbert/distilbert-base-uncased"
    BERT_BASE_UNCASED = "google-bert/bert-base-uncased"
    MODERN_BERT_BASE = "answerdotai/ModernBERT-base"
    MPNET_BASE = "microsoft/mpnet-base"
    CONTRIEVER = "facebook/contriever"
    ALL_MINILM_L6_V2 = "sentence-transformers/all-MiniLM-L6-v2"
    E5_BASE_V2 = "intfloat/e5-base-v2"
    YOUTUBE_BERT = "flboehm/youtube-bert"
    YOUTUBE_BERT_10 = "flboehm/youtube-bert_10"
    T5_SMALL_YOUTUBE = "marianna13/t5-small-finetuned-youtube"
    DEBERTA_V3_XSMALL = "microsoft/deberta-v3-xsmall"
    YOUTUBE_XLM_ROBERTA_BASE = "AmaanP314/youtube-xlm-roberta-base-sentiment-multilingual"

model_selection = ModelType.BERT_TINY
model_name = model_selection.value
test_frac = 0.1
batch_size = 8192

# Channel-level split
channel_ids = norm_df["channel_id"].unique().to_list()
train_ch, test_ch = train_test_split(channel_ids, test_size=test_frac, random_state=42)

train_df = norm_df.filter(pl.col("channel_id").is_in(train_ch))
test_df = norm_df.filter(pl.col("channel_id").is_in(test_ch))

from transformers import AutoTokenizer
text_tokenizer = AutoTokenizer.from_pretrained(model_name)

train_dataset = TitleDataset(train_df, tokenizer=text_tokenizer)
test_dataset = TitleDataset(test_df, tokenizer=text_tokenizer)

from torch.utils.data import DataLoader

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)

model = TitleRegressor(
    model_name=model_name,
    freeze_n_layers=22,
    dropout_rate=0.5
)
    
model.to(device)
model.summary()

NameError: name 'norm_df' is not defined

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
loss_fn = torch.nn.HuberLoss()
epochs = 256

from torch.utils.tensorboard import SummaryWriter
from datetime import datetime

log_dir = f"runs/title_regression_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
writer = SummaryWriter(log_dir=log_dir)

from sklearn.metrics import mean_squared_error, r2_score
from tqdm.notebook import trange

from torch.amp import autocast, GradScaler

scaler = GradScaler()

for epoch in trange(epochs, desc="Training Epochs"):
    model.train()
    total_loss = 0

    for batch in train_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        targets = batch["target"].unsqueeze(1).to(device)

        optimizer.zero_grad()

        with autocast(device_type='cuda'):
            preds = model(input_ids, attention_mask)
            loss = loss_fn(preds, targets)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()

    # Log training loss
    writer.add_scalar("Loss/Train", total_loss / len(train_loader), epoch)

    if (epoch + 1) % 4 == 0:
        model.eval()
        val_preds = []
        val_targets = []
        val_loss = 0

        with torch.no_grad():
            for batch in test_loader:
                input_ids = batch["input_ids"].to(device)
                attention_mask = batch["attention_mask"].to(device)
                targets = batch["target"].unsqueeze(1).to(device)

                with autocast(device_type='cuda'):
                    preds = model(input_ids, attention_mask)
                    loss = loss_fn(preds, targets)

                val_preds.extend(preds.squeeze().cpu().tolist())
                val_targets.extend(targets.squeeze().cpu().tolist())
                val_loss += loss.item()

        val_mse = mean_squared_error(val_targets, val_preds)
        val_r2 = r2_score(val_targets, val_preds)

        # Log validation metrics
        writer.add_scalar("Loss/Val", val_loss / len(test_loader), epoch)
        writer.add_scalar("Metrics/Val_MSE", val_mse, epoch)
        writer.add_scalar("Metrics/Val_R2", val_r2, epoch)

        print(f"Epoch {epoch+1}/{epochs} - "
              f"Train Loss: {total_loss / len(train_loader):.4f} | "
              f"Val Loss: {val_loss / len(test_loader):.4f} | "
              f"Val MSE: {val_mse:.4f} | Val R2: {val_r2:.4f}")

: 

In [ ]:
import os

print(os.getcwd())

from pathlib import Path
import av
import numpy as np
import torch
import polars as pl

# Config
data_path = "../data/video_data.parquet"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load metadata
df = pl.read_parquet(data_path)

df = df.filter(
    pl.col('media_type') == 'video',
    pl.col("view_count").is_not_null(),
    )

# Get unique channel list
channels = df.select("channel_id").unique().to_series()

# List to collect per-channel normalized DataFrames
normalized_chunks = []

# Loop through channels and normalize
for ch in channels:
    
    channel_df = df.filter(pl.col("channel_id") == ch)
    # log scale view_count
    channel_df = channel_df.with_columns(
        pl.col("view_count").log().alias("log_view_count")
    )

    # set target to log view_count
    channel_df = channel_df.with_columns(
        pl.col("log_view_count").alias("target")
    )
    
    # Normalize view count by Z-score
    mean_vc = channel_df.select(pl.col("log_view_count").mean()).item()
    std_vc = channel_df.select(pl.col("log_view_count").std()).item()
    channel_df = channel_df.with_columns(
        ((pl.col("log_view_count") - mean_vc) / std_vc).alias("target")
    )

    # Add mean, std for debug
    channel_df = channel_df.with_columns(
        pl.lit(mean_vc).alias("mean_log_view_count"),
        pl.lit(std_vc).alias("std_log_view_count")
    )

    normalized_chunks.append(channel_df)

# Concatenate all normalized chunks
norm_df = pl.concat(normalized_chunks)
norm_df = norm_df.drop_nulls(subset=["target", "video_title"])


print(norm_df.shape)

f:\Project\YTIntroModel\notebooks
(7713, 28)


In [ ]:
import torch
import torch.nn as nn
from transformers import AutoModel
from torch.utils.data import Dataset

def collate_fn(batch):
    input_ids = torch.stack([item['input_ids'] for item in batch])
    attention_mask = torch.stack([item['attention_mask'] for item in batch])
    targets = torch.stack([item['target'] for item in batch])
    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "target": targets
    }

class TitleDataset(Dataset):
    def __init__(self, df, tokenizer, target_col="target", max_length=32):
        self.targets = df[target_col].to_list()
        self.encodings = tokenizer(
            df["video_title"].to_list(),
            padding="max_length",
            truncation=True,
            max_length=max_length,
            return_tensors="pt"
        )

    def __len__(self):
        return len(self.targets)

    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.encodings.items()}
        item["target"] = torch.tensor(self.targets[idx]).float()
        return item

class TitleRegressor(nn.Module):
    def __init__(self, model_name, dropout_rate=0.2, freeze_n_layers=None):
        super().__init__()
        self.dropout_rate = dropout_rate
        self.encoder = AutoModel.from_pretrained(model_name)

        print(self.encoder.config)

        if freeze_n_layers is not None:
            self._freeze_layers(freeze_n_layers)
        
        hidden_dim = self.encoder.config.hidden_size
        self.regressor = nn.Sequential(
            nn.Linear(hidden_dim, 128),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(128, 1)  # Output layer for regression
        )

    def _freeze_layers(self, n):
        # Freeze embeddings if present
        if hasattr(self.encoder, 'embeddings'):
            for param in self.encoder.embeddings.parameters():
                param.requires_grad = False

        # Try locating layer stack via known patterns
        candidate_paths = [
            ("encoder.layer", getattr(self.encoder, "encoder", None)),
            ("transformer.layer", getattr(self.encoder, "transformer", None)),
            ("distilbert.transformer.layer", getattr(getattr(self.encoder, "distilbert", None), "transformer", None)),
            ("layers", self.encoder),  # ModernBERT: layers at top-level
        ]

        for name, module in candidate_paths:
            if module and hasattr(module, "layer"):
                layers = getattr(module, "layer")
                if isinstance(layers, nn.ModuleList):
                    break
            elif isinstance(module, nn.Module) and hasattr(module, "layers"):
                layers = getattr(module, "layers")
                if isinstance(layers, nn.ModuleList):
                    break
        else:
            # Fallback: search recursively for first ModuleList with enough layers
            layers = next(
                (m for m in self.encoder.modules() if isinstance(m, nn.ModuleList) and len(m) >= n),
                None
            )
            if layers is None:
                raise ValueError("Cannot locate transformer layers in the model")

        # Cap n at available number of layers
        n = min(n, len(layers))
        print(f"Freezing first {n} of {len(layers)} layers.")

        for layer in layers[:n]:
            for param in layer.parameters():
                param.requires_grad = False


    def forward(self, input_ids, attention_mask):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        cls_embedding = outputs.last_hidden_state[:, 0, :]  # CLS token
        return self.regressor(cls_embedding)

    def summary(self):
        print(f"TitleRegressor with {self.encoder.config.hidden_size} hidden size and {len(self.regressor)} layers.")
        total, trainable = 0, 0
        for name, param in self.named_parameters():  # fixed from `model.named_parameters()`
            total += param.numel()
            if param.requires_grad:
                trainable += param.numel()
                print(f"Trainable: {name} - {param.numel()} params")
        print()
        print(f"Total parameters: {total:,}")
        print(f"Trainable parameters: {trainable:,}")
        print(f"Percentage of trainable parameters: {trainable / total:.2%}")
        
        effective_parameters = int(total * (1 - self.dropout_rate) // 1)
        print(f"Effective parameters after dropout: {effective_parameters:,}")

        effective_trainable = int(trainable * (1 - self.dropout_rate) // 1)
        print(f"Effective trainable parameters after dropout: {effective_trainable:,}")

from sklearn.model_selection import train_test_split

from enum import Enum
# Model configuration
class ModelType(Enum):
    BERT_TINY = "prajjwal1/bert-tiny"
    DISTILBERT_BASE = "distilbert/distilbert-base-uncased"
    BERT_BASE_UNCASED = "google-bert/bert-base-uncased"
    MODERN_BERT_BASE = "answerdotai/ModernBERT-base"
    MPNET_BASE = "microsoft/mpnet-base"
    CONTRIEVER = "facebook/contriever"
    ALL_MINILM_L6_V2 = "sentence-transformers/all-MiniLM-L6-v2"
    E5_BASE_V2 = "intfloat/e5-base-v2"
    YOUTUBE_BERT = "flboehm/youtube-bert"
    YOUTUBE_BERT_10 = "flboehm/youtube-bert_10"
    T5_SMALL_YOUTUBE = "marianna13/t5-small-finetuned-youtube"
    DEBERTA_V3_XSMALL = "microsoft/deberta-v3-xsmall"
    YOUTUBE_XLM_ROBERTA_BASE = "AmaanP314/youtube-xlm-roberta-base-sentiment-multilingual"

model_selection = ModelType.BERT_TINY
model_name = model_selection.value
test_frac = 0.1
batch_size = 8192

# Channel-level split
channel_ids = norm_df["channel_id"].unique().to_list()
train_ch, test_ch = train_test_split(channel_ids, test_size=test_frac, random_state=42)

train_df = norm_df.filter(pl.col("channel_id").is_in(train_ch))
test_df = norm_df.filter(pl.col("channel_id").is_in(test_ch))

from transformers import AutoTokenizer
text_tokenizer = AutoTokenizer.from_pretrained(model_name)

train_dataset = TitleDataset(train_df, tokenizer=text_tokenizer)
test_dataset = TitleDataset(test_df, tokenizer=text_tokenizer)

from torch.utils.data import DataLoader

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)

model = TitleRegressor(
    model_name=model_name,
    freeze_n_layers=22,
    dropout_rate=0.5
)
    
model.to(device)
model.summary()

BertConfig {
  "attention_probs_dropout_prob": 0.1,
  "classifier_dropout": null,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 128,
  "initializer_range": 0.02,
  "intermediate_size": 512,
  "layer_norm_eps": 1e-12,
  "max_position_embeddings": 512,
  "model_type": "bert",
  "num_attention_heads": 2,
  "num_hidden_layers": 2,
  "pad_token_id": 0,
  "position_embedding_type": "absolute",
  "torch_dtype": "float32",
  "transformers_version": "4.53.0.dev0",
  "type_vocab_size": 2,
  "use_cache": true,
  "vocab_size": 30522
}

Freezing first 2 of 2 layers.
TitleRegressor with 128 hidden size and 4 layers.
Trainable: encoder.pooler.dense.weight - 16384 params
Trainable: encoder.pooler.dense.bias - 128 params
Trainable: regressor.0.weight - 16384 params
Trainable: regressor.0.bias - 128 params
Trainable: regressor.3.weight - 128 params
Trainable: regressor.3.bias - 1 params

Total parameters: 4,402,561
Trainable parameters: 33,153
Percentage of trainable parameters:

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
loss_fn = torch.nn.HuberLoss()
epochs = 256

from torch.utils.tensorboard import SummaryWriter
from datetime import datetime

log_dir = f"runs/title_regression_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
writer = SummaryWriter(log_dir=log_dir)

from sklearn.metrics import mean_squared_error, r2_score
from tqdm.notebook import trange

from torch.amp import autocast, GradScaler

scaler = GradScaler()

for epoch in trange(epochs, desc="Training Epochs"):
    model.train()
    total_loss = 0

    for batch in train_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        targets = batch["target"].unsqueeze(1).to(device)

        optimizer.zero_grad()

        with autocast(device_type='cuda'):
            preds = model(input_ids, attention_mask)
            loss = loss_fn(preds, targets)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()

    # Log training loss
    writer.add_scalar("Loss/Train", total_loss / len(train_loader), epoch)

    if (epoch + 1) % 4 == 0:
        model.eval()
        val_preds = []
        val_targets = []
        val_loss = 0

        with torch.no_grad():
            for batch in test_loader:
                input_ids = batch["input_ids"].to(device)
                attention_mask = batch["attention_mask"].to(device)
                targets = batch["target"].unsqueeze(1).to(device)

                with autocast(device_type='cuda'):
                    preds = model(input_ids, attention_mask)
                    loss = loss_fn(preds, targets)

                val_preds.extend(preds.squeeze().cpu().tolist())
                val_targets.extend(targets.squeeze().cpu().tolist())
                val_loss += loss.item()

        val_mse = mean_squared_error(val_targets, val_preds)
        val_r2 = r2_score(val_targets, val_preds)

        # Log validation metrics
        writer.add_scalar("Loss/Val", val_loss / len(test_loader), epoch)
        writer.add_scalar("Metrics/Val_MSE", val_mse, epoch)
        writer.add_scalar("Metrics/Val_R2", val_r2, epoch)

        print(f"Epoch {epoch+1}/{epochs} - "
              f"Train Loss: {total_loss / len(train_loader):.4f} | "
              f"Val Loss: {val_loss / len(test_loader):.4f} | "
              f"Val MSE: {val_mse:.4f} | Val R2: {val_r2:.4f}")

Training Epochs:   0%|          | 0/256 [00:00<?, ?it/s]

Epoch 4/256 - Train Loss: 0.4265 | Val Loss: 0.4135 | Val MSE: 0.9903 | Val R2: 0.0020
Epoch 8/256 - Train Loss: 0.4007 | Val Loss: 0.4139 | Val MSE: 0.9836 | Val R2: 0.0086
Epoch 12/256 - Train Loss: 0.3923 | Val Loss: 0.4188 | Val MSE: 0.9878 | Val R2: 0.0045
Epoch 16/256 - Train Loss: 0.3770 | Val Loss: 0.4237 | Val MSE: 0.9965 | Val R2: -0.0044
Epoch 20/256 - Train Loss: 0.3752 | Val Loss: 0.4298 | Val MSE: 1.0089 | Val R2: -0.0168
Epoch 24/256 - Train Loss: 0.3686 | Val Loss: 0.4273 | Val MSE: 1.0052 | Val R2: -0.0131
Epoch 28/256 - Train Loss: 0.3624 | Val Loss: 0.4276 | Val MSE: 1.0083 | Val R2: -0.0162
Epoch 32/256 - Train Loss: 0.3599 | Val Loss: 0.4289 | Val MSE: 1.0134 | Val R2: -0.0213
Epoch 36/256 - Train Loss: 0.3557 | Val Loss: 0.4298 | Val MSE: 1.0170 | Val R2: -0.0249
Epoch 40/256 - Train Loss: 0.3545 | Val Loss: 0.4312 | Val MSE: 1.0198 | Val R2: -0.0278
Epoch 44/256 - Train Loss: 0.3543 | Val Loss: 0.4310 | Val MSE: 1.0194 | Val R2: -0.0274
Epoch 48/256 - Train Loss:

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
loss_fn = torch.nn.HuberLoss()
epochs = 256

from torch.utils.tensorboard import SummaryWriter
from datetime import datetime

log_dir = f"runs/title_regression_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
writer = SummaryWriter(log_dir=log_dir)

from sklearn.metrics import mean_squared_error, r2_score
from tqdm.notebook import trange

from torch.amp import autocast, GradScaler

scaler = GradScaler()

for epoch in trange(epochs, desc="Training Epochs"):
    model.train()
    total_loss = 0

    for batch in train_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        targets = batch["target"].unsqueeze(1).to(device)

        optimizer.zero_grad()

        with autocast(device_type='cuda'):
            preds = model(input_ids, attention_mask)
            loss = loss_fn(preds, targets)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()

    # Log training loss
    writer.add_scalar("Loss/Train", total_loss / len(train_loader), epoch)

    if (epoch + 1) % 4 == 0:
        model.eval()
        val_preds = []
        val_targets = []
        val_loss = 0

        with torch.no_grad():
            for batch in test_loader:
                input_ids = batch["input_ids"].to(device)
                attention_mask = batch["attention_mask"].to(device)
                targets = batch["target"].unsqueeze(1).to(device)

                with autocast(device_type='cuda'):
                    preds = model(input_ids, attention_mask)
                    loss = loss_fn(preds, targets)

                val_preds.extend(preds.squeeze().cpu().tolist())
                val_targets.extend(targets.squeeze().cpu().tolist())
                val_loss += loss.item()

        val_mse = mean_squared_error(val_targets, val_preds)
        val_r2 = r2_score(val_targets, val_preds)

        # Log validation metrics
        writer.add_scalar("Loss/Val", val_loss / len(test_loader), epoch)
        writer.add_scalar("Metrics/Val_MSE", val_mse, epoch)
        writer.add_scalar("Metrics/Val_R2", val_r2, epoch)

        print(f"Epoch {epoch+1}/{epochs} - "
              f"Train Loss: {total_loss / len(train_loader):.4f} | "
              f"Val Loss: {val_loss / len(test_loader):.4f} | "
              f"Val MSE: {val_mse:.4f} | Val R2: {val_r2:.4f}")

Training Epochs:   0%|          | 0/256 [00:00<?, ?it/s]

Epoch 4/256 - Train Loss: 0.3066 | Val Loss: 0.4361 | Val MSE: 1.0393 | Val R2: -0.0474
Epoch 8/256 - Train Loss: 0.3044 | Val Loss: 0.4336 | Val MSE: 1.0329 | Val R2: -0.0410
Epoch 12/256 - Train Loss: 0.3027 | Val Loss: 0.4346 | Val MSE: 1.0345 | Val R2: -0.0426
Epoch 16/256 - Train Loss: 0.3026 | Val Loss: 0.4361 | Val MSE: 1.0389 | Val R2: -0.0471
Epoch 20/256 - Train Loss: 0.3085 | Val Loss: 0.4370 | Val MSE: 1.0407 | Val R2: -0.0489
Epoch 24/256 - Train Loss: 0.3057 | Val Loss: 0.4330 | Val MSE: 1.0300 | Val R2: -0.0381
Epoch 28/256 - Train Loss: 0.3023 | Val Loss: 0.4327 | Val MSE: 1.0295 | Val R2: -0.0376
Epoch 32/256 - Train Loss: 0.3045 | Val Loss: 0.4337 | Val MSE: 1.0322 | Val R2: -0.0403
Epoch 36/256 - Train Loss: 0.3011 | Val Loss: 0.4334 | Val MSE: 1.0313 | Val R2: -0.0394
Epoch 40/256 - Train Loss: 0.3030 | Val Loss: 0.4329 | Val MSE: 1.0297 | Val R2: -0.0378
Epoch 44/256 - Train Loss: 0.3025 | Val Loss: 0.4346 | Val MSE: 1.0341 | Val R2: -0.0422
Epoch 48/256 - Train Lo

In [ ]:
# load the model
model_path = "models/title_regressor_bert_tiny.pt"
if os.path.exists(model_path):
    model.load_state_dict(torch.load(model_path, map_location=device))
    print(f"Model loaded from {model_path}")

Model loaded from models/title_regressor_bert_tiny.pt


In [ ]:
inf_df = test_df.clone()
# create prediction column
inf_df = inf_df.with_columns(
    pl.lit(np.nan).alias("predicted_target")
)

# Inference
model.eval()
predictions = []
with torch.no_grad():
    for batch in test_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)

        with autocast(device_type='cuda'):
            preds = model(input_ids, attention_mask)

        predictions.extend(preds.squeeze().cpu().tolist())
        
# Update the DataFrame with predictions
inf_df = inf_df.with_columns(
    pl.Series(predictions, dtype=pl.Float32).alias("predicted_target")
)

inf_df

video_id,video_title,video_description,media_type,duration,fps,height,width,date_uploaded,timestamp,filesize_approx,chapters,thumbnail,categories,tags,view_count,like_count,comment_count,automatic_captions,subtitles,heatmap,channel_name,channel_handle,channel_id,log_view_count,target,mean_log_view_count,std_log_view_count,predicted_target
str,str,str,str,i64,f64,i64,i64,str,i64,i64,list[struct[3]],str,list[str],list[str],i64,i64,i64,str,str,list[struct[3]],str,str,str,f64,f64,f64,f64,f32
"""-hMr1TeEnNA""","""Lost Media From Nickelodeon 20…","""Start speaking a new language …","""video""",1301,60.0,1080,1920,"""20220813""",1660420800,176516146,"[{0.0,""AD"",65.0}, {65.0,""Intro"",127.0}, … {1113.0,""Oh Dear"",1301.0}]","""https://i.ytimg.com/vi/-hMr1Te…","[""Entertainment""]","[""Blameitonjorge"", ""blame it on jorge"", … ""Theories""]",1387101,54202,2900,"""https://www.youtube.com/api/ti…",null,"[{0.0,13.02,0.619801}, {13.02,26.04,0.114035}, … {1288.98,1302.0,0.09749}]","""blameitonjorge""","""@blameitonjorge""","""UCU3oAyhHDXjgbxazqOEG2UA""",14.142727,-0.129147,14.252996,0.853826,-0.220093
"""-HuEtZH6WDw""","""Top 40 Lost or Banned Episodes…","""As of 2/28/15: 1) LOST 2) h…","""video""",2201,30.0,1080,1920,"""20150228""",1425140536,35619524,"[{0.0,""Intro"",23.0}, {23.0,""The Mask of Matches"",50.0}, … {2137.0,""Outro"",2201.0}]","""https://i.ytimg.com/vi_webp/-H…","[""Entertainment""]","[""Blameitonjorge"", ""lost episode"", … ""Rugrats""]",10586159,115454,14000,"""https://www.youtube.com/api/ti…",null,"[{0.0,22.01,0.820265}, {22.01,44.02,0.303198}, … {2178.99,2201.0,0.221569}]","""blameitonjorge""","""@blameitonjorge""","""UCU3oAyhHDXjgbxazqOEG2UA""",16.175058,2.251116,14.252996,0.853826,0.172607
"""-JQdbaHJZYc""","""7 Insanely Shitty Superhero Ri…","""Spanish Version: https://www.y…","""video""",631,60.0,1080,1920,"""20161106""",1478451600,266353104,"[{0.0,""Intro"",96.0}, {96.0,""Avengers Grimm"",150.0}, … {440.0,""Iron Hero"",631.0}]","""https://i.ytimg.com/vi_webp/-J…","[""Entertainment""]","[""Blameitonjorge"", ""7 Insanely Shitty Superhero Ripoffs"", … ""Mysteries""]",7880456,80960,8800,"""https://www.youtube.com/api/ti…",null,"[{0.0,6.32,1.0}, {6.32,12.64,0.629425}, … {625.68,632.0,0.021482}]","""blameitonjorge""","""@blameitonjorge""","""UCU3oAyhHDXjgbxazqOEG2UA""",15.879896,1.905423,14.252996,0.853826,0.513184
"""-QDNJcMWJiM""","""Top 13 Lost Horror Films | bla…","""Today on blameitonjorge, we're…","""video""",1322,60.0,1080,1920,"""20181027""",1540656000,153399509,"[{0.0,""Intro"",47.0}, {47.0,""ROBBING CLEOPATRA'S TOMB"",115.0}, … {1102.0,""1925 RE-EDIT #1"",1322.0}]","""https://i.ytimg.com/vi_webp/-Q…","[""Entertainment""]","[""Blameitonjorge"", ""blame it on jorge"", … ""Phantom of the Opera""]",674565,17651,1200,"""https://www.youtube.com/api/ti…",null,"[{0.0,13.22,0.158089}, {13.22,26.44,0.043257}, … {1308.78,1322.0,0.0}]","""blameitonjorge""","""@blameitonjorge""","""UCU3oAyhHDXjgbxazqOEG2UA""",13.421823,-0.973468,14.252996,0.853826,0.185669
"""-WVzB_2bgKs""","""10 Episodes of Kids Shows That…","""Today on blameitonjorge, we ta…","""video""",1487,60.0,1080,1920,"""20170819""",1503158401,186897784,"[{0.0,""<Untitled Chapter 1>"",174.0}, {174.0,""Boris is Addicted To Drugs in Captain Planet"",453.0}, … {824.0,""Steven Universe Touches on Abusive Relationships"",1487.0}]","""https://i.ytimg.com/vi_webp/-W…","[""Entertainment""]","[""Captain Planet"", ""Boris"", … ""Mysteries""]",4600211,111612,14000,"""https://www.youtube.com/api/ti…",null,"[{0.0,14.87,1.0}, {14.87,29.74,0.171075}, … {1472.13,1487.0,0.0}]","""blameitonjorge""","""@blameitonjorge""","""UCU3oAyhHDXjgbxazqOEG2UA""",15.341613,1.274986,14.252996,0.853826,0.039307
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""YFJN2bqE-sA""","""5 More Strange and Unnerving Y…","""Tonight we are checking out fi…","""video""",525,30.0,720,1280,"""20170610""",1497068857,37938808,"[{0.0,""Intro"",60.0}, {60.0,""No Description"",232.0}, … {389.0,""The True S

In [ ]:
inf_df = test_df.clone()
# create prediction column
inf_df = inf_df.with_columns(
    pl.lit(np.nan).alias("predicted_target")
)

# Inference
model.eval()
predictions = []
with torch.no_grad():
    for batch in test_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)

        with autocast(device_type='cuda'):
            preds = model(input_ids, attention_mask)

        predictions.extend(preds.squeeze().cpu().tolist())
        
# Update the DataFrame with predictions
inf_df = inf_df.with_columns(
    pl.Series(predictions, dtype=pl.Float32).alias("predicted_target")
)

select_cols = ['video_id', 'video_title', 'duration', 'fps', 'view_count',
               'channel_handle', 'target', 'mean_log_view_count',
               'std_log_view_count', 'predicted_target']

inf_df = inf_df.select(select_cols)

inf_df

video_id,video_title,duration,fps,view_count,channel_handle,target,mean_log_view_count,std_log_view_count,predicted_target
str,str,i64,f64,i64,str,f64,f64,f64,f32
"""-hMr1TeEnNA""","""Lost Media From Nickelodeon 20…",1301,60.0,1387101,"""@blameitonjorge""",-0.129147,14.252996,0.853826,-0.220093
"""-HuEtZH6WDw""","""Top 40 Lost or Banned Episodes…",2201,30.0,10586159,"""@blameitonjorge""",2.251116,14.252996,0.853826,0.172607
"""-JQdbaHJZYc""","""7 Insanely Shitty Superhero Ri…",631,60.0,7880456,"""@blameitonjorge""",1.905423,14.252996,0.853826,0.513184
"""-QDNJcMWJiM""","""Top 13 Lost Horror Films | bla…",1322,60.0,674565,"""@blameitonjorge""",-0.973468,14.252996,0.853826,0.185669
"""-WVzB_2bgKs""","""10 Episodes of Kids Shows That…",1487,60.0,4600211,"""@blameitonjorge""",1.274986,14.252996,0.853826,0.039307
…,…,…,…,…,…,…,…,…,…
"""YFJN2bqE-sA""","""5 More Strange and Unnerving Y…",525,30.0,578863,"""@Nexpo""",-0.970177,14.42901,1.195854,-0.518555
"""Z03BX5IEoZI""","""Disturbing Things from Around …",803,30.0,7060841,"""@Nexpo""",1.121428,14.42901,1.195854,0.654785
"""ZjFBi3otrU8""","""The ""Real Blair Witch"" Inciden…",1899,30.0,1563861,"""@Nexpo""",-0.139099,14.42901,1.195854,0.159424


In [ ]:
inf_df = test_df.clone()
# create prediction column
inf_df = inf_df.with_columns(
    pl.lit(np.nan).alias("predicted_target")
)

# Inference
model.eval()
predictions = []
with torch.no_grad():
    for batch in test_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)

        with autocast(device_type='cuda'):
            preds = model(input_ids, attention_mask)

        predictions.extend(preds.squeeze().cpu().tolist())
        
# Update the DataFrame with predictions
inf_df = inf_df.with_columns(
    pl.Series(predictions, dtype=pl.Float32).alias("predicted_target")
)

select_cols = ['video_id', 'video_title', 'duration', 'fps', 'view_count',
               'channel_handle', 'target', 'mean_log_view_count',
               'std_log_view_count', 'predicted_target', 'duration', 'fps']

inf_df = inf_df.select(select_cols)

inf_df

DuplicateError: the name 'duration' is duplicate

It's possible that multiple expressions are returning the same default column name. If this is the case, try renaming the columns with `.alias("new_name")` to avoid duplicate column names.

In [ ]:
inf_df = test_df.clone()
# create prediction column
inf_df = inf_df.with_columns(
    pl.lit(np.nan).alias("predicted_target")
)

# Inference
model.eval()
predictions = []
with torch.no_grad():
    for batch in test_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)

        with autocast(device_type='cuda'):
            preds = model(input_ids, attention_mask)

        predictions.extend(preds.squeeze().cpu().tolist())
        
# Update the DataFrame with predictions
inf_df = inf_df.with_columns(
    pl.Series(predictions, dtype=pl.Float32).alias("predicted_target")
)

select_cols = ['video_id', 'video_title', 'view_count',
               'channel_handle', 'target', 'mean_log_view_count',
               'std_log_view_count', 'predicted_target']

inf_df = inf_df.select(select_cols)

inf_df

video_id,video_title,view_count,channel_handle,target,mean_log_view_count,std_log_view_count,predicted_target
str,str,i64,str,f64,f64,f64,f32
"""-hMr1TeEnNA""","""Lost Media From Nickelodeon 20…",1387101,"""@blameitonjorge""",-0.129147,14.252996,0.853826,-0.220093
"""-HuEtZH6WDw""","""Top 40 Lost or Banned Episodes…",10586159,"""@blameitonjorge""",2.251116,14.252996,0.853826,0.172607
"""-JQdbaHJZYc""","""7 Insanely Shitty Superhero Ri…",7880456,"""@blameitonjorge""",1.905423,14.252996,0.853826,0.513184
"""-QDNJcMWJiM""","""Top 13 Lost Horror Films | bla…",674565,"""@blameitonjorge""",-0.973468,14.252996,0.853826,0.185669
"""-WVzB_2bgKs""","""10 Episodes of Kids Shows That…",4600211,"""@blameitonjorge""",1.274986,14.252996,0.853826,0.039307
…,…,…,…,…,…,…,…
"""YFJN2bqE-sA""","""5 More Strange and Unnerving Y…",578863,"""@Nexpo""",-0.970177,14.42901,1.195854,-0.518555
"""Z03BX5IEoZI""","""Disturbing Things from Around …",7060841,"""@Nexpo""",1.121428,14.42901,1.195854,0.654785
"""ZjFBi3otrU8""","""The ""Real Blair Witch"" Inciden…",1563861,"""@Nexpo""",-0.139099,14.42901,1.195854,0.159424


In [ ]:
inf_df = test_df.clone()
# create prediction column
inf_df = inf_df.with_columns(
    pl.lit(np.nan).alias("predicted_target")
)

# Inference
model.eval()
predictions = []
with torch.no_grad():
    for batch in test_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)

        with autocast(device_type='cuda'):
            preds = model(input_ids, attention_mask)

        predictions.extend(preds.squeeze().cpu().tolist())
        
# Update the DataFrame with predictions
inf_df = inf_df.with_columns(
    pl.Series(predictions, dtype=pl.Float32).alias("predicted_target")
)

select_cols = ['video_id', 'video_title', 'view_count',
               'channel_handle', 'target', 'mean_log_view_count',
               'std_log_view_count', 'predicted_target']

inf_df = inf_df.select(select_cols)

inf_df = inf_df.with_columns(
    # error
    (pl.col("predicted_target") - pl.col("target")).alias("error"),
).sort("error")


inf_df

video_id,video_title,view_count,channel_handle,target,mean_log_view_count,std_log_view_count,predicted_target,error
str,str,i64,str,f64,f64,f64,f32,f64
"""iHzzSao6ypE""","""The Simple Solution to Traffic""",42141929,"""@CGPGrey""",2.224157,15.418685,0.961204,-0.012161,-2.236318
"""tdOKTV0kPM0""","""Top 20 Saddest Moments In Kid …",16429197,"""@blameitonjorge""",2.765872,14.252996,0.853826,0.539551,-2.226322
"""-HuEtZH6WDw""","""Top 40 Lost or Banned Episodes…",10586159,"""@blameitonjorge""",2.251116,14.252996,0.853826,0.172607,-2.078508
"""vIADc1oMiBE""","""Top 30 Disturbing Moments in K…",7727912,"""@blameitonjorge""",1.88253,14.252996,0.853826,-0.138184,-2.020713
"""3Ztj_4fFJck""","""Dan Schneider: A Scandal at Ni…",7863656,"""@blameitonjorge""",1.902924,14.252996,0.853826,-0.059784,-1.962708
…,…,…,…,…,…,…,…,…
"""u7TlWQ9qdik""","""The Lost ""Uncle Walt"" Short Wa…",284776,"""@blameitonjorge""",-1.983469,14.252996,0.853826,0.236572,2.220041
"""ufsYK3Eecw4""","""Behind the Scenes of TEKOI""",571687,"""@CGPGrey""",-2.249613,15.418685,0.961204,-0.024277,2.225337
"""TftIoyO7ZqE""","""The Mysterious Death of Elisa …",136464,"""@Nexpo""",-2.178523,14.42901,1.195854,0.271973,2.450495


In [ ]:
inf_df = test_df.clone()
# create prediction column
inf_df = inf_df.with_columns(
    pl.lit(np.nan).alias("predicted_target")
)

# Inference
model.eval()
predictions = []
with torch.no_grad():
    for batch in test_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)

        with autocast(device_type='cuda'):
            preds = model(input_ids, attention_mask)

        predictions.extend(preds.squeeze().cpu().tolist())
        
# Update the DataFrame with predictions
inf_df = inf_df.with_columns(
    pl.Series(predictions, dtype=pl.Float32).alias("predicted_target")
)

select_cols = ['video_id', 'video_title', 'view_count',
               'channel_handle', 'target', 'mean_log_view_count',
               'std_log_view_count', 'predicted_target']

inf_df = inf_df.select(select_cols)

inf_df = inf_df.with_columns(
    # error
    (pl.col("predicted_target") - pl.col("target")).abs().alias("error"),
).sort("error")


inf_df

video_id,video_title,view_count,channel_handle,target,mean_log_view_count,std_log_view_count,predicted_target,error
str,str,i64,str,f64,f64,f64,f32,f64
"""xUyVEax17iA""","""The ""Where Did This Image Come…",1723275,"""@blameitonjorge""",0.125015,14.252996,0.853826,0.124146,0.00087
"""TCyuiJeyG2c""","""Tea Time Analysis""",1201150,"""@Nexpo""",-0.35976,14.42901,1.195854,-0.352539,0.007221
"""Que2myhwr48""","""Investigating the ""Magic World…",1867036,"""@Nexpo""",0.009075,14.42901,1.195854,0.001497,0.007578
"""wAUGFiLrplg""","""""My Sister"" Analysis""",1519661,"""@Nexpo""",-0.163074,14.42901,1.195854,-0.153931,0.009143
"""bhyYgnhhKFw""","""The True Cost of the Royal Fam…",6731369,"""@CGPGrey""",0.315858,15.418685,0.961204,0.304443,0.011415
…,…,…,…,…,…,…,…,…
"""tdOKTV0kPM0""","""Top 20 Saddest Moments In Kid …",16429197,"""@blameitonjorge""",2.765872,14.252996,0.853826,0.539551,2.226322
"""iHzzSao6ypE""","""The Simple Solution to Traffic""",42141929,"""@CGPGrey""",2.224157,15.418685,0.961204,-0.012161,2.236318
"""TftIoyO7ZqE""","""The Mysterious Death of Elisa …",136464,"""@Nexpo""",-2.178523,14.42901,1.195854,0.271973,2.450495


In [ ]:
inf_df = test_df.clone()
# create prediction column
inf_df = inf_df.with_columns(
    pl.lit(np.nan).alias("predicted_target")
)

# Inference
model.eval()
predictions = []
with torch.no_grad():
    for batch in test_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)

        with autocast(device_type='cuda'):
            preds = model(input_ids, attention_mask)

        predictions.extend(preds.squeeze().cpu().tolist())
        
# Update the DataFrame with predictions
inf_df = inf_df.with_columns(
    pl.Series(predictions, dtype=pl.Float32).alias("predicted_target")
)

select_cols = ['video_id', 'video_title', 'view_count',
               'channel_handle', 'target', 'mean_log_view_count',
               'std_log_view_count', 'predicted_target']

inf_df = inf_df.select(select_cols)

inf_df = inf_df.with_columns(
    # error
    (pl.col("predicted_target") - pl.col("target")).abs().alias("error"),
).sort("error")

inf_df = inf_df.with_columns(
    # predicted view count
    (pl.col("predicted_target").exp() * pl.col("std_log_view_count") + pl.col("mean_log_view_count")).alias("predicted_view_count"),
)

inf_df

video_id,video_title,view_count,channel_handle,target,mean_log_view_count,std_log_view_count,predicted_target,error,predicted_view_count
str,str,i64,str,f64,f64,f64,f32,f64,f64
"""xUyVEax17iA""","""The ""Where Did This Image Come…",1723275,"""@blameitonjorge""",0.125015,14.252996,0.853826,0.124146,0.00087,15.219682
"""TCyuiJeyG2c""","""Tea Time Analysis""",1201150,"""@Nexpo""",-0.35976,14.42901,1.195854,-0.352539,0.007221,15.269577
"""Que2myhwr48""","""Investigating the ""Magic World…",1867036,"""@Nexpo""",0.009075,14.42901,1.195854,0.001497,0.007578,15.626656
"""wAUGFiLrplg""","""""My Sister"" Analysis""",1519661,"""@Nexpo""",-0.163074,14.42901,1.195854,-0.153931,0.009143,15.454253
"""bhyYgnhhKFw""","""The True Cost of the Royal Fam…",6731369,"""@CGPGrey""",0.315858,15.418685,0.961204,0.304443,0.011415,16.721953
…,…,…,…,…,…,…,…,…,…
"""tdOKTV0kPM0""","""Top 20 Saddest Moments In Kid …",16429197,"""@blameitonjorge""",2.765872,14.252996,0.853826,0.539551,2.226322,15.71751
"""iHzzSao6ypE""","""The Simple Solution to Traffic""",42141929,"""@CGPGrey""",2.224157,15.418685,0.961204,-0.012161,2.236318,16.368271
"""TftIoyO7ZqE""","""The Mysterious Death of Elisa …",136464,"""@Nexpo""",-2.178523,14.42901,1.195854,0.271973,2.450495,15.998629


In [ ]:
inf_df = test_df.clone()
# create prediction column
inf_df = inf_df.with_columns(
    pl.lit(np.nan).alias("predicted_target")
)

# Inference
model.eval()
predictions = []
with torch.no_grad():
    for batch in test_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)

        with autocast(device_type='cuda'):
            preds = model(input_ids, attention_mask)

        predictions.extend(preds.squeeze().cpu().tolist())
        
# Update the DataFrame with predictions
inf_df = inf_df.with_columns(
    pl.Series(predictions, dtype=pl.Float32).alias("predicted_target")
)

select_cols = ['video_id', 'video_title', 'view_count',
               'channel_handle', 'target', 'mean_log_view_count',
               'std_log_view_count', 'predicted_target']

inf_df = inf_df.select(select_cols)

inf_df = inf_df.with_columns(
    # error
    (pl.col("predicted_target") - pl.col("target")).abs().alias("error"),
).sort("error")

inf_df = inf_df.with_columns(
    # predicted view count
    (pl.col("predicted_target") * pl.col("std_log_view_count") + pl.col("mean_log_view_count")).alias("predicted_view_count"),
)

inf_df

video_id,video_title,view_count,channel_handle,target,mean_log_view_count,std_log_view_count,predicted_target,error,predicted_view_count
str,str,i64,str,f64,f64,f64,f32,f64,f64
"""xUyVEax17iA""","""The ""Where Did This Image Come…",1723275,"""@blameitonjorge""",0.125015,14.252996,0.853826,0.124146,0.00087,14.358995
"""TCyuiJeyG2c""","""Tea Time Analysis""",1201150,"""@Nexpo""",-0.35976,14.42901,1.195854,-0.352539,0.007221,14.007425
"""Que2myhwr48""","""Investigating the ""Magic World…",1867036,"""@Nexpo""",0.009075,14.42901,1.195854,0.001497,0.007578,14.430801
"""wAUGFiLrplg""","""""My Sister"" Analysis""",1519661,"""@Nexpo""",-0.163074,14.42901,1.195854,-0.153931,0.009143,14.244932
"""bhyYgnhhKFw""","""The True Cost of the Royal Fam…",6731369,"""@CGPGrey""",0.315858,15.418685,0.961204,0.304443,0.011415,15.711317
…,…,…,…,…,…,…,…,…,…
"""tdOKTV0kPM0""","""Top 20 Saddest Moments In Kid …",16429197,"""@blameitonjorge""",2.765872,14.252996,0.853826,0.539551,2.226322,14.713679
"""iHzzSao6ypE""","""The Simple Solution to Traffic""",42141929,"""@CGPGrey""",2.224157,15.418685,0.961204,-0.012161,2.236318,15.406995
"""TftIoyO7ZqE""","""The Mysterious Death of Elisa …",136464,"""@Nexpo""",-2.178523,14.42901,1.195854,0.271973,2.450495,14.75425


In [ ]:
inf_df = test_df.clone()
# create prediction column
inf_df = inf_df.with_columns(
    pl.lit(np.nan).alias("predicted_target")
)

# Inference
model.eval()
predictions = []
with torch.no_grad():
    for batch in test_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)

        with autocast(device_type='cuda'):
            preds = model(input_ids, attention_mask)

        predictions.extend(preds.squeeze().cpu().tolist())
        
# Update the DataFrame with predictions
inf_df = inf_df.with_columns(
    pl.Series(predictions, dtype=pl.Float32).alias("predicted_target")
)

select_cols = ['video_id', 'video_title', 'view_count',
               'channel_handle', 'target', 'mean_log_view_count',
               'std_log_view_count', 'predicted_target']

inf_df = inf_df.select(select_cols)

inf_df = inf_df.with_columns(
    # error
    (pl.col("predicted_target") - pl.col("target")).abs().alias("error"),
).sort("error")

inf_df = inf_df.with_columns(
    # predicted view count
    ((pl.col("predicted_target") * pl.col("std_log_view_count") + pl.col("mean_log_view_count"))).exp().alias("predicted_view_count"),
)

inf_df

video_id,video_title,view_count,channel_handle,target,mean_log_view_count,std_log_view_count,predicted_target,error,predicted_view_count
str,str,i64,str,f64,f64,f64,f32,f64,f64
"""xUyVEax17iA""","""The ""Where Did This Image Come…",1723275,"""@blameitonjorge""",0.125015,14.252996,0.853826,0.124146,0.00087,1.7220e6
"""TCyuiJeyG2c""","""Tea Time Analysis""",1201150,"""@Nexpo""",-0.35976,14.42901,1.195854,-0.352539,0.007221,1.2116e6
"""Que2myhwr48""","""Investigating the ""Magic World…",1867036,"""@Nexpo""",0.009075,14.42901,1.195854,0.001497,0.007578,1.8502e6
"""wAUGFiLrplg""","""""My Sister"" Analysis""",1519661,"""@Nexpo""",-0.163074,14.42901,1.195854,-0.153931,0.009143,1.5364e6
"""bhyYgnhhKFw""","""The True Cost of the Royal Fam…",6731369,"""@CGPGrey""",0.315858,15.418685,0.961204,0.304443,0.011415,6.6579e6
…,…,…,…,…,…,…,…,…,…
"""tdOKTV0kPM0""","""Top 20 Saddest Moments In Kid …",16429197,"""@blameitonjorge""",2.765872,14.252996,0.853826,0.539551,2.226322,2.4551e6
"""iHzzSao6ypE""","""The Simple Solution to Traffic""",42141929,"""@CGPGrey""",2.224157,15.418685,0.961204,-0.012161,2.236318,4.9110e6
"""TftIoyO7ZqE""","""The Mysterious Death of Elisa …",136464,"""@Nexpo""",-2.178523,14.42901,1.195854,0.271973,2.450495,2.5568e6


In [ ]:
import torch
import polars as pl
import numpy as np
from sklearn.cluster import KMeans
from sklearn.metrics import r2_score
from torch.nn.functional import cosine_similarity

# Ensure model exposes embedding extraction
def get_cls_embeddings(model, test_loader, device="cuda"):
    model.eval()
    cls_embeddings = []
    with torch.no_grad():
        for batch in test_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            outputs = model.encoder(input_ids=input_ids, attention_mask=attention_mask)
            cls = outputs.last_hidden_state[:, 0, :]
            cls_embeddings.append(cls.cpu())
    return torch.cat(cls_embeddings, dim=0).numpy()

In [ ]:
inf_df = inf_df.with_columns(
    ((pl.col("predicted_target") * pl.col("std_log_view_count") + pl.col("mean_log_view_count"))).exp().alias("predicted_view_count"),
)

In [ ]:
# Step 1: Extract CLS Embeddings
print("[CLS] Extracting CLS token embeddings from test set...")
cls_embeddings = get_cls_embeddings(model, test_loader, device)
print(f"[CLS] Shape: {cls_embeddings.shape}")

[CLS] Extracting CLS token embeddings from test set...
[CLS] Shape: (385, 128)


In [ ]:
# Step 2: Run KMeans Clustering
n_clusters = 20  # change if needed
print(f"[KMeans] Clustering into {n_clusters} clusters...")
kmeans = KMeans(n_clusters=n_clusters, random_state=42)
cluster_ids = kmeans.fit_predict(cls_embeddings)

inf_df = inf_df.with_columns(
    pl.Series(cluster_ids).alias("cluster_id")
)

[KMeans] Clustering into 20 clusters...


In [ ]:
# Step 3: Per-Cluster Metrics
print("\n=== Per-Cluster Performance ===")
for cid in sorted(set(cluster_ids)):
    cluster = inf_df.filter(pl.col("cluster_id") == cid)
    if len(cluster) < 2:
        print(f"Cluster {cid:>2}: Too few samples ({len(cluster)})")
        continue
    y_true = cluster["target"].to_numpy()
    y_pred = cluster["predicted_target"].to_numpy()
    mean_err = cluster["error"].mean()
    r2 = r2_score(y_true, y_pred)
    print(f"Cluster {cid:>2}: R² = {r2:.4f} | Mean Error = {mean_err:.3f} | N = {len(cluster)}")


=== Per-Cluster Performance ===
Cluster  0: R² = -0.9347 | Mean Error = 0.502 | N = 3
Cluster  1: R² = 0.3052 | Mean Error = 0.704 | N = 28
Cluster  2: R² = 0.4503 | Mean Error = 0.238 | N = 31
Cluster  3: R² = 0.2451 | Mean Error = 0.606 | N = 26
Cluster  4: R² = 0.8048 | Mean Error = 0.146 | N = 33
Cluster  5: R² = 0.0998 | Mean Error = 0.563 | N = 13
Cluster  6: R² = 0.1728 | Mean Error = 0.736 | N = 33
Cluster  7: R² = -0.1259 | Mean Error = 0.637 | N = 13
Cluster  8: R² = 0.4607 | Mean Error = 0.300 | N = 10
Cluster  9: R² = -0.1294 | Mean Error = 1.558 | N = 2
Cluster 10: R² = 0.1766 | Mean Error = 0.852 | N = 25
Cluster 11: R² = -0.0311 | Mean Error = 0.788 | N = 12
Cluster 12: R² = 0.1447 | Mean Error = 0.556 | N = 20
Cluster 13: R² = 0.1373 | Mean Error = 1.229 | N = 17
Cluster 14: R² = -0.0012 | Mean Error = 1.063 | N = 9
Cluster 15: R² = 0.2946 | Mean Error = 0.616 | N = 25
Cluster 16: R² = 0.0636 | Mean Error = 1.501 | N = 16
Cluster 17: R² = -0.0801 | Mean Error = 0.436 |

In [ ]:
# Step 4: Intra-Cluster Neighbor Inspection
print("\n=== Intra-Cluster Nearest Neighbor Error Inspection ===")
titles = inf_df["video_title"].to_list()
errors = inf_df["error"].to_list()
clusters = inf_df["cluster_id"].to_list()
emb_tensor = torch.tensor(cls_embeddings).float()

for i in range(min(3, len(emb_tensor))):  # adjust range as needed
    cid = clusters[i]
    indices_in_cluster = [j for j, c in enumerate(clusters) if c == cid and j != i]
    if not indices_in_cluster:
        continue

    query = emb_tensor[i].unsqueeze(0)
    cluster_embs = emb_tensor[indices_in_cluster]
    sims = cosine_similarity(query, cluster_embs).cpu().numpy().flatten()
    topk_idx = sims.argsort()[-3:][::-1]
    real_indices = [indices_in_cluster[j] for j in topk_idx]

    print(f"\n▶ {titles[i]} (err = {errors[i]:.3f}) [Cluster {cid}]")
    for j in real_indices:
        print(f"  → {titles[j]} (err = {errors[j]:.3f})")


=== Intra-Cluster Nearest Neighbor Error Inspection ===

▶ The "Where Did This Image Come From?" Iceberg (err = 0.001) [Cluster 17]
  → Death & Dynasties (err = 0.128)
  → Explaining Ratafak Plachta | blameitonjorge (err = 0.156)
  → Should all locks have keys? Phones, Castles, Encryption, and You. (err = 0.176)

▶ Tea Time Analysis (err = 0.007) [Cluster 4]
  → Ten Years Later (err = 0.294)
  → The True Cost of the Royal Family Explained (err = 0.011)
  → Where is Scandinavia? (err = 0.223)

▶ Investigating the "Magic World of Ania” (err = 0.008) [Cluster 4]
  → YouTube vs Grey: A Ballad of Accidental Suspension (err = 0.064)
  → The Lost Media Gameshow Iceberg | blameitonjorge (err = 0.123)
  → What is "This Room Does Not Exist"? (err = 0.146)


In [ ]:
# Step 4: Intra-Cluster Neighbor Inspection
print("\n=== Intra-Cluster Nearest Neighbor Error Inspection ===")
titles = inf_df["video_title"].to_list()
errors = inf_df["error"].to_list()
clusters = inf_df["cluster_id"].to_list()
emb_tensor = torch.tensor(cls_embeddings).float()

for i in range(min(10, len(emb_tensor))):  # adjust range as needed
    cid = clusters[i]
    indices_in_cluster = [j for j, c in enumerate(clusters) if c == cid and j != i]
    if not indices_in_cluster:
        continue

    query = emb_tensor[i].unsqueeze(0)
    cluster_embs = emb_tensor[indices_in_cluster]
    sims = cosine_similarity(query, cluster_embs).cpu().numpy().flatten()
    topk_idx = sims.argsort()[-3:][::-1]
    real_indices = [indices_in_cluster[j] for j in topk_idx]

    print(f"\n▶ {titles[i]} (err = {errors[i]:.3f}) [Cluster {cid}]")
    for j in real_indices:
        print(f"  → {titles[j]} (err = {errors[j]:.3f})")


=== Intra-Cluster Nearest Neighbor Error Inspection ===

▶ The "Where Did This Image Come From?" Iceberg (err = 0.001) [Cluster 17]
  → Death & Dynasties (err = 0.128)
  → Explaining Ratafak Plachta | blameitonjorge (err = 0.156)
  → Should all locks have keys? Phones, Castles, Encryption, and You. (err = 0.176)

▶ Tea Time Analysis (err = 0.007) [Cluster 4]
  → Ten Years Later (err = 0.294)
  → The True Cost of the Royal Family Explained (err = 0.011)
  → Where is Scandinavia? (err = 0.223)

▶ Investigating the "Magic World of Ania” (err = 0.008) [Cluster 4]
  → YouTube vs Grey: A Ballad of Accidental Suspension (err = 0.064)
  → The Lost Media Gameshow Iceberg | blameitonjorge (err = 0.123)
  → What is "This Room Does Not Exist"? (err = 0.146)

▶ "My Sister" Analysis (err = 0.009) [Cluster 4]
  → 2020 Vision: The YouTube Channel Predicting Death (err = 0.079)
  → What is "This Room Does Not Exist"? (err = 0.146)
  → What is a Leap Year? (err = 0.057)

▶ The True Cost of the Royal Fa

In [ ]:
print("\n=== Cluster Content Summaries ===")

num_examples_per_cluster = 5  # adjust if needed
sort_by_error = True  # set False to sample randomly

for cid in sorted(set(cluster_ids)):
    cluster = inf_df.filter(pl.col("cluster_id") == cid)
    n = len(cluster)
    if n < 2:
        print(f"\nCluster {cid:>2}: Too small ({n} samples)")
        continue

    mean_err = cluster["error"].mean()
    std_err = cluster["error"].std()
    print(f"\n📦 Cluster {cid} — {n} samples")
    print(f"   Mean Abs Error = {mean_err:.3f} ± {std_err:.3f}")

    if sort_by_error:
        rows = cluster.sort("error", descending=True).head(num_examples_per_cluster)
        label = "Top Error Titles"
    else:
        rows = cluster.sample(n=min(num_examples_per_cluster, n))


=== Cluster Content Summaries ===

📦 Cluster 0 — 3 samples
   Mean Abs Error = 0.502 ± 0.165

📦 Cluster 1 — 28 samples
   Mean Abs Error = 0.704 ± 0.409

📦 Cluster 2 — 31 samples
   Mean Abs Error = 0.238 ± 0.183

📦 Cluster 3 — 26 samples
   Mean Abs Error = 0.606 ± 0.593

📦 Cluster 4 — 33 samples
   Mean Abs Error = 0.146 ± 0.100

📦 Cluster 5 — 13 samples
   Mean Abs Error = 0.563 ± 0.300

📦 Cluster 6 — 33 samples
   Mean Abs Error = 0.736 ± 0.395

📦 Cluster 7 — 13 samples
   Mean Abs Error = 0.637 ± 0.222

📦 Cluster 8 — 10 samples
   Mean Abs Error = 0.300 ± 0.368

📦 Cluster 9 — 2 samples
   Mean Abs Error = 1.558 ± 0.510

📦 Cluster 10 — 25 samples
   Mean Abs Error = 0.852 ± 0.520

📦 Cluster 11 — 12 samples
   Mean Abs Error = 0.788 ± 0.481

📦 Cluster 12 — 20 samples
   Mean Abs Error = 0.556 ± 0.163

📦 Cluster 13 — 17 samples
   Mean Abs Error = 1.229 ± 0.586

📦 Cluster 14 — 9 samples
   Mean Abs Error = 1.063 ± 0.612

📦 Cluster 15 — 25 samples
   Mean Abs Error = 0.616 ± 0.345

📦

In [ ]:
print("\n=== Cluster Content Summaries ===")

num_examples_per_cluster = 5  # adjust if needed
sort_by_error = True  # set False to sample randomly

for cid in sorted(set(cluster_ids)):
    cluster = inf_df.filter(pl.col("cluster_id") == cid)
    n = len(cluster)
    if n < 2:
        print(f"\nCluster {cid:>2}: Too small ({n} samples)")
        continue

    mean_err = cluster["error"].mean()
    std_err = cluster["error"].std()
    print(f"\n📦 Cluster {cid} — {n} samples")
    print(f"   Mean Abs Error = {mean_err:.3f} ± {std_err:.3f}")

    if sort_by_error:
        rows = cluster.sort("error", descending=True).head(num_examples_per_cluster)
        label = "Top Error Titles"
    else:
        rows = cluster.sample(n=min(num_examples_per_cluster, n))
        label = "Random Titles"

    print(f"   {label}:")
    for r in rows.iter_rows(named=True):
        print(f"     - \"{r['video_title']}\" (err={r['error']:.3f})")


=== Cluster Content Summaries ===

📦 Cluster 0 — 3 samples
   Mean Abs Error = 0.502 ± 0.165
   Top Error Titles:
     - "10 Most WTF Characters in Courage the Cowardly Dog | blameitonjorge" (err=0.692)
     - "Las Vegas isn't Las Vegas" (err=0.410)
     - "The Battle of SHARKS!" (err=0.403)

📦 Cluster 1 — 28 samples
   Mean Abs Error = 0.704 ± 0.409
   Top Error Titles:
     - "9 Episodes of Kids Shows That Dealt With Serious Issues | blameitonjorge" (err=1.825)
     - "Gemini and the End of the World" (err=1.381)
     - "One Year & 100,000 Subscribers Later (Thank You!)" (err=1.352)
     - "7 Unnecessarily Horrifying Characters in Kid-Friendly Games (N64) | blameitonjorge" (err=1.219)
     - "Erratas: The Mysterious Word You Shouldn't Search For | blameitonjorge" (err=1.068)

📦 Cluster 2 — 31 samples
   Mean Abs Error = 0.238 ± 0.183
   Top Error Titles:
     - "The Woman Who Went Missing Three Times: Hannah Upp | blameitonjorge" (err=0.813)
     - "10 Japanese Rituals That Might Sc

In [ ]:
print("\n=== Cluster Content Summaries ===")

num_examples_per_cluster = 5  # adjust if needed
sort_by_error = True  # set False to sample randomly

for cid in sorted(set(cluster_ids)):
    cluster = inf_df.filter(pl.col("cluster_id") == cid)
    n = len(cluster)
    if n < 2:
        print(f"\nCluster {cid:>2}: Too small ({n} samples)")
        continue

    mean_err = cluster["error"].mean()
    std_err = cluster["error"].std()
    print(f"\n📦 Cluster {cid} — {n} samples")
    print(f"   Mean Abs Error = {mean_err:.3f} ± {std_err:.3f}")

    if sort_by_error:
        rows = cluster.sort("error", descending=True).head(num_examples_per_cluster)
        label = "Top Error Titles"
    else:
        rows = cluster.sample(n=min(num_examples_per_cluster, n))
        label = "Random Titles"

    print(f"   {label}:")
    for r in rows.iter_rows(named=True):
        print(f"     - \"{r['video_title']}\" (err={r['error']:.3f})")


=== Cluster Content Summaries ===

📦 Cluster 0 — 3 samples
   Mean Abs Error = 0.502 ± 0.165
   Top Error Titles:
     - "10 Most WTF Characters in Courage the Cowardly Dog | blameitonjorge" (err=0.692)
     - "Las Vegas isn't Las Vegas" (err=0.410)
     - "The Battle of SHARKS!" (err=0.403)

📦 Cluster 1 — 28 samples
   Mean Abs Error = 0.704 ± 0.409
   Top Error Titles:
     - "9 Episodes of Kids Shows That Dealt With Serious Issues | blameitonjorge" (err=1.825)
     - "Gemini and the End of the World" (err=1.381)
     - "One Year & 100,000 Subscribers Later (Thank You!)" (err=1.352)
     - "7 Unnecessarily Horrifying Characters in Kid-Friendly Games (N64) | blameitonjorge" (err=1.219)
     - "Erratas: The Mysterious Word You Shouldn't Search For | blameitonjorge" (err=1.068)

📦 Cluster 2 — 31 samples
   Mean Abs Error = 0.238 ± 0.183
   Top Error Titles:
     - "The Woman Who Went Missing Three Times: Hannah Upp | blameitonjorge" (err=0.813)
     - "10 Japanese Rituals That Might Sc

In [ ]:
print("\n=== Cluster Content Summaries ===")

num_examples_per_cluster = 5  # adjust if needed
sort_by_error = True  # set False to sample randomly

for cid in sorted(set(cluster_ids)):
    cluster = inf_df.filter(pl.col("cluster_id") == cid)
    n = len(cluster)
    if n < 2:
        print(f"\nCluster {cid:>2}: Too small ({n} samples)")
        continue

    mean_err = cluster["error"].mean()
    std_err = cluster["error"].std()
    print(f"\n📦 Cluster {cid} — {n} samples")
    print(f"   Mean Abs Error = {mean_err:.3f} ± {std_err:.3f}")

    if sort_by_error:
        rows = cluster.sort("error", descending=True).head(num_examples_per_cluster)
        label = "Top Error Titles"
    else:
        rows = cluster.sample(n=min(num_examples_per_cluster, n))
        label = "Random Titles"

    print(f"   {label}:")
    for r in rows.iter_rows(named=True):
        print(f"     - \"{r['video_title']}\" (err={r['error']:.3f})")


=== Cluster Content Summaries ===

📦 Cluster 0 — 3 samples
   Mean Abs Error = 0.502 ± 0.165
   Top Error Titles:
     - "10 Most WTF Characters in Courage the Cowardly Dog | blameitonjorge" (err=0.692)
     - "Las Vegas isn't Las Vegas" (err=0.410)
     - "The Battle of SHARKS!" (err=0.403)

📦 Cluster 1 — 28 samples
   Mean Abs Error = 0.704 ± 0.409
   Top Error Titles:
     - "9 Episodes of Kids Shows That Dealt With Serious Issues | blameitonjorge" (err=1.825)
     - "Gemini and the End of the World" (err=1.381)
     - "One Year & 100,000 Subscribers Later (Thank You!)" (err=1.352)
     - "7 Unnecessarily Horrifying Characters in Kid-Friendly Games (N64) | blameitonjorge" (err=1.219)
     - "Erratas: The Mysterious Word You Shouldn't Search For | blameitonjorge" (err=1.068)

📦 Cluster 2 — 31 samples
   Mean Abs Error = 0.238 ± 0.183
   Top Error Titles:
     - "The Woman Who Went Missing Three Times: Hannah Upp | blameitonjorge" (err=0.813)
     - "10 Japanese Rituals That Might Sc

In [ ]:
print("\n=== Cluster Content Summaries ===")

num_examples_per_cluster = 5  # adjust if needed
sort_by_error = True  # set False to sample randomly

for cid in sorted(set(cluster_ids)):
    cluster = inf_df.filter(pl.col("cluster_id") == cid)
    n = len(cluster)
    if n < 2:
        print(f"\nCluster {cid:>2}: Too small ({n} samples)")
        continue

    mean_err = cluster["error"].mean()
    std_err = cluster["error"].std()
    print(f"\n📦 Cluster {cid} — {n} samples")
    print(f"   Mean Abs Error = {mean_err:.3f} ± {std_err:.3f}")

    if sort_by_error:
        rows = cluster.sort("error", descending=True).head(num_examples_per_cluster)
        label = "Top Error Titles"
    else:
        rows = cluster.sample(n=min(num_examples_per_cluster, n))
        label = "Random Titles"

    print(f"   {label}:")
    for r in rows.iter_rows(named=True):
        print(f"     - \"{r['video_title']}\" (err={r['error']:.3f})")


# Optional: print titles closest to the cluster centroid
def top_titles_by_centroid(cls_embeddings, cluster_ids, emb_tensor, inf_df, cid, k=5):
    indices = [i for i, c in enumerate(cluster_ids) if c == cid]
    cluster_embs = emb_tensor[indices]
    centroid = cluster_embs.mean(dim=0, keepdim=True)
    sims = cosine_similarity(centroid, cluster_embs).squeeze().cpu().numpy()
    top_k = sims.argsort()[-k:][::-1]
    return [inf_df["video_title"][indices[i]] for i in top_k]

print("\n=== Cluster Summaries (Representative Titles) ===")
for cid in sorted(set(cluster_ids)):
    cluster = inf_df.filter(pl.col("cluster_id") == cid)
    if len(cluster) < 2:
        continue
    print(f"\n📦 Cluster {cid} — {len(cluster)} samples")
    reps = top_titles_by_centroid(cls_embeddings, cluster_ids, emb_tensor, inf_df, cid, k=5)
    for title in reps:
        print(f"  - {title}")


=== Cluster Content Summaries ===

📦 Cluster 0 — 3 samples
   Mean Abs Error = 0.502 ± 0.165
   Top Error Titles:
     - "10 Most WTF Characters in Courage the Cowardly Dog | blameitonjorge" (err=0.692)
     - "Las Vegas isn't Las Vegas" (err=0.410)
     - "The Battle of SHARKS!" (err=0.403)

📦 Cluster 1 — 28 samples
   Mean Abs Error = 0.704 ± 0.409
   Top Error Titles:
     - "9 Episodes of Kids Shows That Dealt With Serious Issues | blameitonjorge" (err=1.825)
     - "Gemini and the End of the World" (err=1.381)
     - "One Year & 100,000 Subscribers Later (Thank You!)" (err=1.352)
     - "7 Unnecessarily Horrifying Characters in Kid-Friendly Games (N64) | blameitonjorge" (err=1.219)
     - "Erratas: The Mysterious Word You Shouldn't Search For | blameitonjorge" (err=1.068)

📦 Cluster 2 — 31 samples
   Mean Abs Error = 0.238 ± 0.183
   Top Error Titles:
     - "The Woman Who Went Missing Three Times: Hannah Upp | blameitonjorge" (err=0.813)
     - "10 Japanese Rituals That Might Sc

In [ ]:
print("\n=== Cluster Content Summaries ===")

num_examples_per_cluster = 5  # adjust if needed
sort_by_error = True  # set False to sample randomly

for cid in sorted(set(cluster_ids)):
    cluster = inf_df.filter(pl.col("cluster_id") == cid)
    n = len(cluster)
    if n < 2:
        print(f"\nCluster {cid:>2}: Too small ({n} samples)")
        continue

    mean_err = cluster["error"].mean()
    std_err = cluster["error"].std()
    print(f"\n📦 Cluster {cid} — {n} samples")
    print(f"   Mean Abs Error = {mean_err:.3f} ± {std_err:.3f}")

    if sort_by_error:
        rows = cluster.sort("error", descending=True).head(num_examples_per_cluster)
        label = "Top Error Titles"
    else:
        rows = cluster.sample(n=min(num_examples_per_cluster, n))
        label = "Random Titles"

    print(f"   {label}:")
    for r in rows.iter_rows(named=True):
        print(f"     - \"{r['video_title']}\" (err={r['error']:.3f})")


# Optional: print titles closest to the cluster centroid
def top_titles_by_centroid(cls_embeddings, cluster_ids, emb_tensor, inf_df, cid, k=5):
    indices = [i for i, c in enumerate(cluster_ids) if c == cid]
    cluster_embs = emb_tensor[indices]
    centroid = cluster_embs.mean(dim=0, keepdim=True)
    sims = cosine_similarity(centroid, cluster_embs).squeeze().cpu().numpy()
    top_k = sims.argsort()[-k:][::-1]
    return [inf_df["video_title"][indices[i]] for i in top_k]

print("\n=== Cluster Summaries (Representative Titles) ===")
for cid in sorted(set(cluster_ids)):
    cluster = inf_df.filter(pl.col("cluster_id") == cid)
    if len(cluster) < 2:
        continue
    print(f"\n📦 Cluster {cid} — {len(cluster)} samples")
    reps = top_titles_by_centroid(cls_embeddings, cluster_ids, emb_tensor, inf_df, cid, k=5)
    for title in reps:
        print(f"  - {title}")


=== Cluster Content Summaries ===

📦 Cluster 0 — 3 samples
   Mean Abs Error = 0.502 ± 0.165
   Top Error Titles:
     - "10 Most WTF Characters in Courage the Cowardly Dog | blameitonjorge" (err=0.692)
     - "Las Vegas isn't Las Vegas" (err=0.410)
     - "The Battle of SHARKS!" (err=0.403)

📦 Cluster 1 — 28 samples
   Mean Abs Error = 0.704 ± 0.409
   Top Error Titles:
     - "9 Episodes of Kids Shows That Dealt With Serious Issues | blameitonjorge" (err=1.825)
     - "Gemini and the End of the World" (err=1.381)
     - "One Year & 100,000 Subscribers Later (Thank You!)" (err=1.352)
     - "7 Unnecessarily Horrifying Characters in Kid-Friendly Games (N64) | blameitonjorge" (err=1.219)
     - "Erratas: The Mysterious Word You Shouldn't Search For | blameitonjorge" (err=1.068)

📦 Cluster 2 — 31 samples
   Mean Abs Error = 0.238 ± 0.183
   Top Error Titles:
     - "The Woman Who Went Missing Three Times: Hannah Upp | blameitonjorge" (err=0.813)
     - "10 Japanese Rituals That Might Sc

In [ ]:
,

''

In [ ]:
print("\n=== Cluster Content Summaries ===")

num_examples_per_cluster = 5  # adjust if needed
sort_by_error = True  # set False to sample randomly

for cid in sorted(set(cluster_ids)):
    cluster = inf_df.filter(pl.col("cluster_id") == cid)
    n = len(cluster)
    if n < 2:
        print(f"\nCluster {cid:>2}: Too small ({n} samples)")
        continue

    mean_err = cluster["error"].mean()
    std_err = cluster["error"].std()
    print(f"\n📦 Cluster {cid} — {n} samples")
    print(f"   Mean Abs Error = {mean_err:.3f} ± {std_err:.3f}")

    if sort_by_error:
        rows = cluster.sort("error", descending=True).head(num_examples_per_cluster)
        label = "Top Error Titles"
    else:
        rows = cluster.sample(n=min(num_examples_per_cluster, n))
        label = "Random Titles"

    print(f"   {label}:")
    for r in rows.iter_rows(named=True):
        print(f"     - \"{r['video_title']}\" (err={r['error']:.3f})")


# Optional: print titles closest to the cluster centroid
def top_titles_by_centroid(cls_embeddings, cluster_ids, emb_tensor, inf_df, cid, k=5):
    indices = [i for i, c in enumerate(cluster_ids) if c == cid]
    cluster_embs = emb_tensor[indices]
    centroid = cluster_embs.mean(dim=0, keepdim=True)
    sims = cosine_similarity(centroid, cluster_embs).squeeze().cpu().numpy()
    top_k = sims.argsort()[-k:][::-1]
    return [inf_df["video_title"][indices[i]] for i in top_k]

print("\n=== Cluster Summaries (Representative Titles) ===")
for cid in sorted(set(cluster_ids)):
    cluster = inf_df.filter(pl.col("cluster_id") == cid)
    if len(cluster) < 2:
        continue
    print(f"\n📦 Cluster {cid} — {len(cluster)} samples")
    reps = top_titles_by_centroid(cls_embeddings, cluster_ids, emb_tensor, inf_df, cid, k=5)
    for title in reps:
        print(f"  - {title}")
from torch.nn.functional import cosine_similarity

print("\n=== Cluster Insights ===")

emb_tensor = torch.tensor(cls_embeddings).float()
titles = inf_df["video_title"].to_list()
errors = inf_df["error"].to_list()
clusters = inf_df["cluster_id"].to_list()
targets = inf_df["target"].to_list()
preds = inf_df["predicted_target"].to_list()

for cid in sorted(set(clusters)):
    cluster_idx = [i for i, c in enumerate(clusters) if c == cid]
    if len(cluster_idx) < 2:
        print(f"\nCluster {cid:>2}: Too few samples ({len(cluster_idx)})")
        continue

    y_true = [targets[i] for i in cluster_idx]
    y_pred = [preds[i] for i in cluster_idx]
    r2 = r2_score(y_true, y_pred)
    mean_err = np.mean([errors[i] for i in cluster_idx])
    std_err = np.std([errors[i] for i in cluster_idx])

    print(f"\n📦 Cluster {cid} — {len(cluster_idx)} samples")
    print(f"   R² = {r2:.4f} | Mean Abs Error = {mean_err:.3f} ± {std_err:.3f}")

    # Top-3 most erroneous
    top_err = sorted(cluster_idx, key=lambda i: -errors[i])[:3]
    print("   ❌ Worst Predictions:")
    for i in top_err:
        print(f"     - {titles[i]} (err = {errors[i]:.3f})")

    # Top-3 most accurate
    low_err = sorted(cluster_idx, key=lambda i: errors[i])[:3]
    print("   ✅ Most Accurate Predictions:")
    for i in low_err:
        print(f"     - {titles[i]} (err = {errors[i]:.3f})")

    # Top-3 most representative (closest to cluster centroid)
    cluster_embs = emb_tensor[cluster_idx]
    centroid = cluster_embs.mean(dim=0, keepdim=True)
    sims = cosine_similarity(centroid, cluster_embs).squeeze().cpu().numpy()
    top_centroid = np.argsort(sims)[-3:][::-1]
    print("   🎯 Most Representative Titles:")
    for i in top_centroid:
        real_idx = cluster_idx[i]
        print(f"     - {titles[real_idx]} (err = {errors[real_idx]:.3f})")


=== Cluster Content Summaries ===

📦 Cluster 0 — 3 samples
   Mean Abs Error = 0.502 ± 0.165
   Top Error Titles:
     - "10 Most WTF Characters in Courage the Cowardly Dog | blameitonjorge" (err=0.692)
     - "Las Vegas isn't Las Vegas" (err=0.410)
     - "The Battle of SHARKS!" (err=0.403)

📦 Cluster 1 — 28 samples
   Mean Abs Error = 0.704 ± 0.409
   Top Error Titles:
     - "9 Episodes of Kids Shows That Dealt With Serious Issues | blameitonjorge" (err=1.825)
     - "Gemini and the End of the World" (err=1.381)
     - "One Year & 100,000 Subscribers Later (Thank You!)" (err=1.352)
     - "7 Unnecessarily Horrifying Characters in Kid-Friendly Games (N64) | blameitonjorge" (err=1.219)
     - "Erratas: The Mysterious Word You Shouldn't Search For | blameitonjorge" (err=1.068)

📦 Cluster 2 — 31 samples
   Mean Abs Error = 0.238 ± 0.183
   Top Error Titles:
     - "The Woman Who Went Missing Three Times: Hannah Upp | blameitonjorge" (err=0.813)
     - "10 Japanese Rituals That Might Sc

In [ ]:
from torch.nn.functional import cosine_similarity
from sklearn.metrics import r2_score

print("\n=== Cluster Summary by Centroid ===")

emb_tensor = torch.tensor(cls_embeddings).float()
titles = inf_df["video_title"].to_list()
errors = inf_df["error"].to_list()
clusters = inf_df["cluster_id"].to_list()
targets = inf_df["target"].to_list()
preds = inf_df["predicted_target"].to_list()

num_titles = 5  # number of representative titles to show

for cid in sorted(set(clusters)):
    cluster_idx = [i for i, c in enumerate(clusters) if c == cid]
    if len(cluster_idx) < 2:
        print(f"\nCluster {cid:>2}: Too few samples ({len(cluster_idx)})")
        continue

    # Compute R² and mean error
    y_true = [targets[i] for i in cluster_idx]
    y_pred = [preds[i] for i in cluster_idx]
    try:
        r2 = r2_score(y_true, y_pred)
    except Exception:
        r2 = float("nan")
    mean_err = np.mean([errors[i] for i in cluster_idx])
    std_err = np.std([errors[i] for i in cluster_idx])

    # Compute centroid-based similarity
    cluster_embs = emb_tensor[cluster_idx]
    centroid = cluster_embs.mean(dim=0, keepdim=True)
    sims = cosine_similarity(centroid, cluster_embs).squeeze().cpu().numpy()
    top_idxs = np.argsort(sims)[-num_titles:][::-1]

    print(f"\n📦 Cluster {cid} — {len(cluster_idx)} samples")
    print(f"   R² = {r2:.4f} | Mean Abs Error = {mean_err:.3f} ± {std_err:.3f}")
    print(f"   🎯 Most Representative Titles:")
    for i in top_idxs:
        real_idx = cluster_idx[i]
        print(f"     - {titles[real_idx]} (err = {errors[real_idx]:.3f})")


=== Cluster Summary by Centroid ===

📦 Cluster 0 — 3 samples
   R² = -0.9347 | Mean Abs Error = 0.502 ± 0.134
   🎯 Most Representative Titles:
     - Las Vegas isn't Las Vegas (err = 0.410)
     - 10 Most WTF Characters in Courage the Cowardly Dog | blameitonjorge (err = 0.692)
     - The Battle of SHARKS! (err = 0.403)

📦 Cluster 1 — 28 samples
   R² = 0.3052 | Mean Abs Error = 0.704 ± 0.401
   🎯 Most Representative Titles:
     - Vatican City Explained (err = 0.983)
     - Top 10 Best Episodes of Goosebumps | blameitonjorge (err = 0.027)
     - 12 Lost Pieces of Cartoon Network Media | blameitonjorge (err = 0.866)
     - Gemini and the End of the World (err = 1.381)
     - The European Union Explained* (err = 1.023)

📦 Cluster 2 — 31 samples
   R² = 0.4503 | Mean Abs Error = 0.238 ± 0.180
   🎯 Most Representative Titles:
     - Hi I'm Mary Mary: Explained (err = 0.239)
     - What If the Electoral College is Tied? (err = 0.195)
     - The Cipher Hunt: Gravity Falls' Insane Worldwide

In [ ]:
from torch.nn.functional import cosine_similarity
from sklearn.metrics import r2_score
import numpy as np

print("\n=== Cluster Summary by Centroid (Sorted by R²) ===")

emb_tensor = torch.tensor(cls_embeddings).float()
titles = inf_df["video_title"].to_list()
errors = inf_df["error"].to_list()
clusters = inf_df["cluster_id"].to_list()
targets = inf_df["target"].to_list()
preds = inf_df["predicted_target"].to_list()

num_titles = 5  # number of representative titles to print
cluster_stats = []

# Step 1: Precompute R² and summary stats for all clusters
for cid in sorted(set(clusters)):
    cluster_idx = [i for i, c in enumerate(clusters) if c == cid]
    if len(cluster_idx) < 2:
        continue

    y_true = [targets[i] for i in cluster_idx]
    y_pred = [preds[i] for i in cluster_idx]
    try:
        r2 = r2_score(y_true, y_pred)
    except Exception:
        r2 = float("nan")

    mean_err = np.mean([errors[i] for i in cluster_idx])
    std_err = np.std([errors[i] for i in cluster_idx])

    cluster_stats.append({
        "cid": cid,
        "r2": r2,
        "mean_err": mean_err,
        "std_err": std_err,
        "indices": cluster_idx
    })

# Step 2: Sort clusters by R² descending
cluster_stats = sorted(cluster_stats, key=lambda x: x["r2"], reverse=True)

# Step 3: Print results
for entry in cluster_stats:
    cid = entry["cid"]
    r2 = entry["r2"]
    mean_err = entry["mean_err"]
    std_err = entry["std_err"]
    cluster_idx = entry["indices"]

    # Find representative titles via centroid similarity
    cluster_embs = emb_tensor[cluster_idx]
    centroid = cluster_embs.mean(dim=0, keepdim=True)
    sims = cosine_similarity(centroid, cluster_embs).squeeze().cpu().numpy()
    top_idxs = np.argsort(sims)[-num_titles:][::-1]

    print(f"\n📦 Cluster {cid} — {len(cluster_idx)} samples")
    print(f"   R² = {r2:.4f} | Mean Abs Error = {mean_err:.3f} ± {std_err:.3f}")
    print(f"   🎯 Most Representative Titles:")
    for i in top_idxs:
        real_idx = cluster_idx[i]
        print(f"     - {titles[real_idx]} (err = {errors[real_idx]:.3f})")


=== Cluster Summary by Centroid (Sorted by R²) ===

📦 Cluster 4 — 33 samples
   R² = 0.8048 | Mean Abs Error = 0.146 ± 0.099
   🎯 Most Representative Titles:
     - What is "This Room Does Not Exist"? (err = 0.146)
     - Ten Years Later (err = 0.294)
     - Tea Time Analysis (err = 0.007)
     - Groundhog Day Explained (err = 0.208)
     - The True Cost of the Royal Family Explained (err = 0.011)

📦 Cluster 8 — 10 samples
   R² = 0.4607 | Mean Abs Error = 0.300 ± 0.349
   🎯 Most Representative Titles:
     - Google Translate’s Strange Responses (err = 0.210)
     - Gas Stations and Sonic Sound Effects - Why? (err = 0.059)
     - The Lost Media of Creepypastas (err = 0.204)
     - Top 40 Creepiest Japanese Urban Legends (2017) | blameitonjorge (err = 0.300)
     - Lost Media From Nickelodeon 2022 (err = 0.091)

📦 Cluster 2 — 31 samples
   R² = 0.4503 | Mean Abs Error = 0.238 ± 0.180
   🎯 Most Representative Titles:
     - Hi I'm Mary Mary: Explained (err = 0.239)
     - What If the El

In [ ]:
from torch.nn.functional import cosine_similarity
from sklearn.metrics import r2_score
import numpy as np

print("\n=== Cluster Summary by Centroid (Sorted by R², All Titles) ===")

emb_tensor = torch.tensor(cls_embeddings).float()
titles = inf_df["video_title"].to_list()
errors = inf_df["error"].to_list()
clusters = inf_df["cluster_id"].to_list()
targets = inf_df["target"].to_list()
preds = inf_df["predicted_target"].to_list()

cluster_stats = []

# Step 1: Compute cluster metrics
for cid in sorted(set(clusters)):
    cluster_idx = [i for i, c in enumerate(clusters) if c == cid]
    if len(cluster_idx) < 2:
        continue

    y_true = [targets[i] for i in cluster_idx]
    y_pred = [preds[i] for i in cluster_idx]
    try:
        r2 = r2_score(y_true, y_pred)
    except Exception:
        r2 = float("nan")

    mean_err = np.mean([errors[i] for i in cluster_idx])
    std_err = np.std([errors[i] for i in cluster_idx])

    cluster_stats.append({
        "cid": cid,
        "r2": r2,
        "mean_err": mean_err,
        "std_err": std_err,
        "indices": cluster_idx
    })

# Step 2: Sort clusters by R²
cluster_stats = sorted(cluster_stats, key=lambda x: x["r2"], reverse=True)

# Step 3: Print all titles in each cluster (sorted by proximity to centroid)
for entry in cluster_stats:
    cid = entry["cid"]
    r2 = entry["r2"]
    mean_err = entry["mean_err"]
    std_err = entry["std_err"]
    cluster_idx = entry["indices"]

    cluster_embs = emb_tensor[cluster_idx]
    centroid = cluster_embs.mean(dim=0, keepdim=True)
    sims = cosine_similarity(centroid, cluster_embs).squeeze().cpu().numpy()
    sorted_by_similarity = np.argsort(sims)[::-1]  # highest similarity first

    print(f"\n📦 Cluster {cid} — {len(cluster_idx)} samples")
    print(f"   R² = {r2:.4f} | Mean Abs Error = {mean_err:.3f} ± {std_err:.3f}")
    print(f"   🎯 Titles (most representative first):")
    for i in sorted_by_similarity:
        real_idx = cluster_idx[i]
        print(f"     - {titles[real_idx]} (err = {errors[real_idx]:.3f})")


=== Cluster Summary by Centroid (Sorted by R², All Titles) ===

📦 Cluster 4 — 33 samples
   R² = 0.8048 | Mean Abs Error = 0.146 ± 0.099
   🎯 Titles (most representative first):
     - What is "This Room Does Not Exist"? (err = 0.146)
     - Ten Years Later (err = 0.294)
     - Tea Time Analysis (err = 0.007)
     - Groundhog Day Explained (err = 0.208)
     - The True Cost of the Royal Family Explained (err = 0.011)
     - Canada Gets Rid of the Penny (Huzzah!) (err = 0.238)
     - Disturbing Things from Around the Internet [Vol. 2] (err = 0.216)
     - 2020 Vision: The YouTube Channel Predicting Death (err = 0.079)
     - Ben Drowns Again (err = 0.135)
     - The Trouble with Transporters (err = 0.251)
     - 'Indian' or 'Native American'?  [Reservations, Part 0] (err = 0.281)
     - Unnerving YouTube Channels #1 (err = 0.047)
     - The Search For A Day With Spongebob Squarepants: The Complete History | blameitonjorge (err = 0.210)
     - Where is Scandinavia? (err = 0.223)
     - 